In [47]:
import pandas as pd
import numpy as np

In [48]:
candidate_emb_df = pd.read_parquet("/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/Embedding/data_outputs/10_candidate_embeddings.parquet")
job_emb_df = pd.read_parquet("/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/Embedding/data_outputs/11_job_embeddings.parquet")

print("Candidate shape:", candidate_emb_df.shape)
print("Job shape:", job_emb_df.shape)

print(candidate_emb_df.columns.tolist())
print(job_emb_df.columns.tolist())

Candidate shape: (20, 5)
Job shape: (80, 5)
['candidate_id', 'candidate_text', 'embedding_model', 'embedding_dim', 'embedding_vector']
['job_id', 'job_text', 'embedding_model', 'embedding_dim', 'embedding_vector']


In [49]:
candidate_emb_df.head(2)

,candidate_id,candidate_text,embedding_model,embedding_dim,embedding_vector
0,C001,query: Candidate group: Software Development. ...,intfloat/multilingual-e5-base,768,"[0.004771376959979534, 0.05155104398727417, -0..."
1,C002,query: Candidate group: Software Development. ...,intfloat/multilingual-e5-base,768,"[0.01015084981918335, 0.04290420189499855, -0...."


In [50]:
job_emb_df.head(2)

,job_id,job_text,embedding_model,embedding_dim,embedding_vector
0,J001,passage: Job title: Angular Frontend Engineer....,intfloat/multilingual-e5-base,768,"[-0.00839162990450859, 0.06643076241016388, 0...."
1,J002,passage: Job title: Next.js Web Developer. Job...,intfloat/multilingual-e5-base,768,"[-0.008345445618033409, 0.07151442766189575, 0..."


In [51]:
candidate_ids = candidate_emb_df["candidate_id"].tolist()
job_ids = job_emb_df["job_id"].tolist()

candidate_matrix = np.vstack(candidate_emb_df["embedding_vector"].apply(np.array).values)
job_matrix = np.vstack(job_emb_df["embedding_vector"].apply(np.array).values)

print("Candidate matrix shape:", candidate_matrix.shape)
print("Job matrix shape:", job_matrix.shape)

Candidate matrix shape: (20, 768)
Job matrix shape: (80, 768)


In [52]:
similarity_matrix = candidate_matrix @ job_matrix.T

print("Similarity matrix shape:", similarity_matrix.shape)
similarity_matrix

Similarity matrix shape: (20, 80)


array([[0.86440044, 0.89392489, 0.87129138, ..., 0.78678668, 0.79478575,
        0.78476273],
       [0.83978236, 0.84947694, 0.84023944, ..., 0.78204534, 0.79673084,
        0.77825782],
       [0.79339123, 0.81362492, 0.79943168, ..., 0.77595479, 0.78989011,
        0.76816995],
       ...,
       [0.80929789, 0.81354954, 0.81343044, ..., 0.7849936 , 0.79237182,
        0.80107705],
       [0.82502147, 0.8465112 , 0.82994007, ..., 0.78249798, 0.80811549,
        0.79413492],
       [0.79494234, 0.82354522, 0.79649621, ..., 0.77947328, 0.77710722,
        0.76342254]])

In [53]:
rows = []

for i, candidate_id in enumerate(candidate_ids):
    for j, job_id in enumerate(job_ids):
        rows.append({
            "candidate_id": candidate_id,
            "job_id": job_id,
            "semantic_similarity": float(similarity_matrix[i, j])
        })

semantic_df = pd.DataFrame(rows)
semantic_df.head(10)

# chưa sort

,candidate_id,job_id,semantic_similarity
0,C001,J001,0.864400
1,C001,J002,0.893925
2,C001,J003,0.871291
3,C001,J004,0.886809
4,C001,J005,0.906228
5,C001,J006,0.880208
6,C001,J007,0.869554
7,C001,J008,0.857281
8,C001,J009,0.887343
9,C001,J010,0.857376


In [54]:
semantic_df = semantic_df.sort_values(
    by=["candidate_id", "semantic_similarity"],
    ascending=[True, False]
).reset_index(drop=True)

semantic_df.head(20)

# đã sort

,candidate_id,job_id,semantic_similarity
0,C001,J005,0.906228
1,C001,J002,0.893925
2,C001,J009,0.887343
3,C001,J004,0.886809
4,C001,J006,0.880208
5,C001,J003,0.871291
6,C001,J053,0.870919
7,C001,J007,0.869554
8,C001,J015,0.866792
9,C001,J011,0.865707


In [55]:
semantic_df["semantic_rank"] = semantic_df.groupby("candidate_id").cumcount() + 1
semantic_df.head(20)

,candidate_id,job_id,semantic_similarity,semantic_rank
0,C001,J005,0.906228,1
1,C001,J002,0.893925,2
2,C001,J009,0.887343,3
3,C001,J004,0.886809,4
4,C001,J006,0.880208,5
5,C001,J003,0.871291,6
6,C001,J053,0.870919,7
7,C001,J007,0.869554,8
8,C001,J015,0.866792,9
9,C001,J011,0.865707,10


In [56]:
semantic_df.to_parquet("/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/Embedding/data_outputs/12_candidate_job_semantic_similarity.parquet", index=False)
semantic_df.to_excel("/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/Embedding/data_outputs/12_candidate_job_semantic_similarity.xlsx", index=False)

print("Saved -> 12_candidate_job_semantic_similarity.parquet")
print("Saved -> 12_candidate_job_semantic_similarity.xlsx")

Saved -> 12_candidate_job_semantic_similarity.parquet
Saved -> 12_candidate_job_semantic_similarity.xlsx


In [57]:
# Preview Top-20 semantic results for each candidate
# Note: the export cell above already saves ALL candidate-job pairs.
# This cell is only for viewing/checking Top-20.

TOP_K = 20

top20_semantic = semantic_df[semantic_df["semantic_rank"] <= TOP_K].copy()
top20_semantic = top20_semantic.sort_values(
    ["candidate_id", "semantic_rank"],
    ascending=[True, True]
)

print("Top-K:", TOP_K)
print("Rows in full semantic_df:", len(semantic_df))
print("Rows in top20_semantic:", len(top20_semantic))
print("Candidates:", semantic_df["candidate_id"].nunique())
print("Jobs:", semantic_df["job_id"].nunique())

top20_semantic

Top-K: 20
Rows in full semantic_df: 1600
Rows in top20_semantic: 400
Candidates: 20
Jobs: 80


,candidate_id,job_id,semantic_similarity,semantic_rank
0,C001,J005,0.906228,1
1,C001,J002,0.893925,2
2,C001,J009,0.887343,3
3,C001,J004,0.886809,4
4,C001,J006,0.880208,5
...,...,...,...,...
1535,C020,J076,0.839230,16
1536,C020,J027,0.836302,17
1537,C020,J026,0.835486,18
1538,C020,J014,0.834675,19


1. Đọc output parquet từ file 02:
   - candidate embeddings
   - job embeddings

2. Lấy cột embedding_vector ra.

3. Ép danh sách vector thành ma trận numpy:
   - candidate_matrix
   - job_matrix

4. Tính semantic similarity bằng nhân ma trận:
   similarity_matrix = candidate_matrix @ job_matrix.T

5. Chuyển ma trận similarity thành bảng:
   candidate_id | job_id | semantic_similarity

6. Sắp xếp theo từng candidate:
   candidate_id tăng dần
   semantic_similarity giảm dần

7. Gán semantic_rank:
   job nào giống nghĩa nhất với candidate thì rank 1.

8. Lưu output semantic similarity để file hybrid dùng tiếp.